# Phase 5 — Pandas

Pandas is the main library used for tabular data analysis in this project.

In this phase, we will work directly with the Airline Flight Delay dataset to cover:

- Series and DataFrame
- Indexing and selection
- Filtering
- Sorting
- Adding and modifying columns
- Missing-value analysis and treatment
- Data-type conversion
- Duplicate handling
- String operations
- Date operations
- GroupBy
- Aggregation
- Multiple aggregations
- Merging
- Joining analytical tables
- Reshaping
- Pivot tables
- Business-rule derived columns
- Data validation
- Saving the final cleaned dataset

The raw dataset will be preserved in `df`.
The cleaned dataset will be stored separately in `cleaned_df`.

## 5.1 Pandas Series

A Series is a one-dimensional labeled data structure.

We will extract the `ARR_DELAY` column from the actual airline dataset and inspect it as a Series.

In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd

# Define the absolute source dataset path
file_path = r"F:\Data science\intern all file\project\Airlines project final intern\data\raw_data\flight_delay.csv"

# Load the complete dataset
df = pd.read_csv(file_path)
df.shape


(3000000, 32)

In [3]:
arr_delay_series = df["ARR_DELAY"]

arr_delay_series.head()

0   -14.0
1    -5.0
2     0.0
3    24.0
4    -1.0
Name: ARR_DELAY, dtype: float64

In [4]:
type(arr_delay_series)

pandas.Series

## 5.2 Pandas DataFrame

A DataFrame is a two-dimensional tabular data structure containing rows and columns.

The complete Airline Flight Delay dataset is stored in the DataFrame `df`.

In [5]:
type(df), df.shape

(pandas.DataFrame, (3000000, 32))

## 5.3 Selecting Multiple Columns

We can select multiple business-related columns from the DataFrame.

For airline operational analysis, we will select airline, route, delay, cancellation, and diversion information.

In [6]:
operational_data = df[
    [
        "AIRLINE",
        "ORIGIN",
        "DEST",
        "DEP_DELAY",
        "ARR_DELAY",
        "CANCELLED",
        "DIVERTED"
    ]
]

operational_data.head()

,AIRLINE,ORIGIN,DEST,DEP_DELAY,ARR_DELAY,CANCELLED,DIVERTED
0,United Air Lines Inc.,FLL,EWR,-4.0,-14.0,0.0,0.0
1,Delta Air Lines Inc.,MSP,SEA,-6.0,-5.0,0.0,0.0
2,United Air Lines Inc.,DEN,MSP,6.0,0.0,0.0,0.0
3,Delta Air Lines Inc.,MSP,SFO,-1.0,24.0,0.0,0.0
4,Spirit Air Lines,MCO,DFW,-2.0,-1.0,0.0,0.0


## 5.4 Label-Based Selection with `loc`

`loc` is used to select rows and columns using labels.

We will use it to inspect selected airline records.

In [7]:
df.loc[
    df["AIRLINE"] == "Delta Air Lines Inc.",
    ["AIRLINE", "ORIGIN", "DEST", "DEP_DELAY", "ARR_DELAY"]
].head()

,AIRLINE,ORIGIN,DEST,DEP_DELAY,ARR_DELAY
1,Delta Air Lines Inc.,MSP,SEA,-6.0,-5.0
3,Delta Air Lines Inc.,MSP,SFO,-1.0,24.0
10,Delta Air Lines Inc.,ATL,BDL,69.0,60.0
13,Delta Air Lines Inc.,BDL,ATL,-5.0,-32.0
16,Delta Air Lines Inc.,STT,ATL,-2.0,-9.0


## 5.5 Position-Based Selection with `iloc`

`iloc` selects rows and columns based on their integer positions.

In [8]:
df.iloc[:5, :5]

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416


## 5.6 Advanced Filtering

Pandas allows multiple conditions to be combined.

We will identify flights that:

- had an arrival delay greater than 30 minutes
- were not cancelled
- were not diverted

In [9]:
delayed_operational_flights = df[
    (df["ARR_DELAY"] > 30) &
    (df["CANCELLED"] == 0) &
    (df["DIVERTED"] == 0)
]

delayed_operational_flights.shape

(331071, 32)

## 5.7 Filtering with `isin()`

`isin()` is useful when we want to filter multiple categorical values at once.

We will examine flights operated by selected major airlines.

In [10]:
selected_airlines = [
    "Delta Air Lines Inc.",
    "American Airlines Inc.",
    "United Air Lines Inc."
]

major_airline_flights = df[
    df["AIRLINE"].isin(selected_airlines)
]

major_airline_flights.shape

(1032849, 32)

## 5.8 Range Filtering with `between()`

`between()` makes range-based filtering easier.

We will identify flights with arrival delays between 30 and 60 minutes.

In [11]:
moderate_arrival_delay = df[
    df["ARR_DELAY"].between(30, 60, inclusive="both")
]

moderate_arrival_delay.shape

(164557, 32)

## 5.9 Sorting Data

Sorting helps identify the highest and lowest operational performance records.

We will sort flights by arrival delay.

In [12]:
highest_arrival_delays = df[
    ["AIRLINE", "ORIGIN", "DEST", "ARR_DELAY"]
].sort_values(
    "ARR_DELAY",
    ascending=False
)

highest_arrival_delays.head(10)

,AIRLINE,ORIGIN,DEST,ARR_DELAY
2639771,American Airlines Inc.,DSM,CLT,2934.0
2210421,American Airlines Inc.,VPS,DFW,2913.0
2606159,American Airlines Inc.,TPA,PHL,2912.0
2026734,American Airlines Inc.,STL,ORD,2911.0
2097337,American Airlines Inc.,BFL,PHX,2900.0
1899200,American Airlines Inc.,MEM,CLT,2900.0
518982,American Airlines Inc.,MSN,DFW,2685.0
1541563,American Airlines Inc.,HNL,ORD,2568.0
1383097,American Airlines Inc.,BNA,MIA,2565.0
1976065,Endeavor Air Inc.,CLT,LGA,2560.0


# 5.10 Creating the Cleaning Dataset

The raw DataFrame `df` will not be modified directly.

We will create a separate copy called `cleaned_df`.

This protects the original raw dataset and allows us to compare the data before and after cleaning.

In [13]:
cleaned_df = df.copy()

cleaned_df.shape

(3000000, 32)

## 5.11 Standardizing Column Names

Standardized column names make the dataset easier to work with.

We will:
- remove leading/trailing spaces
- convert names to uppercase
- replace spaces with underscores

In [14]:
cleaned_df.columns = (
    cleaned_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

cleaned_df.columns.tolist()

['FL_DATE',
 'AIRLINE',
 'AIRLINE_DOT',
 'AIRLINE_CODE',
 'DOT_CODE',
 'FL_NUMBER',
 'ORIGIN',
 'ORIGIN_CITY',
 'DEST',
 'DEST_CITY',
 'CRS_DEP_TIME',
 'DEP_TIME',
 'DEP_DELAY',
 'TAXI_OUT',
 'WHEELS_OFF',
 'WHEELS_ON',
 'TAXI_IN',
 'CRS_ARR_TIME',
 'ARR_TIME',
 'ARR_DELAY',
 'CANCELLED',
 'CANCELLATION_CODE',
 'DIVERTED',
 'CRS_ELAPSED_TIME',
 'ELAPSED_TIME',
 'AIR_TIME',
 'DISTANCE',
 'DELAY_DUE_CARRIER',
 'DELAY_DUE_WEATHER',
 'DELAY_DUE_NAS',
 'DELAY_DUE_SECURITY',
 'DELAY_DUE_LATE_AIRCRAFT']

## 5.12 Inspecting Data Types

Correct data types are important for analysis.

For example:
- `FL_DATE` should be a date
- delay and distance fields should be numeric
- airline and airport fields should be textual

In [15]:
cleaned_df.dtypes

FL_DATE                        str
AIRLINE                        str
AIRLINE_DOT                    str
AIRLINE_CODE                   str
DOT_CODE                     int64
FL_NUMBER                    int64
ORIGIN                         str
ORIGIN_CITY                    str
DEST                           str
DEST_CITY                      str
CRS_DEP_TIME                 int64
DEP_TIME                   float64
DEP_DELAY                  float64
TAXI_OUT                   float64
WHEELS_OFF                 float64
WHEELS_ON                  float64
TAXI_IN                    float64
CRS_ARR_TIME                 int64
ARR_TIME                   float64
ARR_DELAY                  float64
CANCELLED                  float64
CANCELLATION_CODE              str
DIVERTED                   float64
CRS_ELAPSED_TIME           float64
ELAPSED_TIME               float64
AIR_TIME                   float64
DISTANCE                   float64
DELAY_DUE_CARRIER          float64
DELAY_DUE_WEATHER   

## 5.13 Converting Flight Date

`FL_DATE` will be converted into a Pandas datetime type.

Invalid date values will become missing values instead of crashing the process.

In [16]:
cleaned_df["FL_DATE"] = pd.to_datetime(
    cleaned_df["FL_DATE"],
    errors="coerce"
)

cleaned_df["FL_DATE"].dtype

dtype('<M8[us]')

## 5.14 Cleaning Text Columns

Text columns may contain unwanted leading or trailing spaces.

We will standardize the main categorical columns while preserving their meaning.

In [17]:
text_columns = [
    "AIRLINE",
    "AIRLINE_DOT",
    "AIRLINE_CODE",
    "ORIGIN",
    "ORIGIN_CITY",
    "DEST",
    "DEST_CITY",
    "CANCELLATION_CODE"
]

for column in text_columns:
    cleaned_df[column] = (
        cleaned_df[column]
        .astype("string")
        .str.strip()
    )

cleaned_df[text_columns].head()

,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CANCELLATION_CODE
0,United Air Lines Inc.,United Air Lines Inc.: UA,UA,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",<NA>
1,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,MSP,"Minneapolis, MN",SEA,"Seattle, WA",<NA>
2,United Air Lines Inc.,United Air Lines Inc.: UA,UA,DEN,"Denver, CO",MSP,"Minneapolis, MN",<NA>
3,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",<NA>
4,Spirit Air Lines,Spirit Air Lines: NK,NK,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",<NA>


## 5.15 Duplicate Records

Duplicate records can distort flight counts and business KPIs.

We will first measure duplicates before removing anything.

In [18]:
duplicate_count_before = cleaned_df.duplicated().sum()

duplicate_count_before

np.int64(0)

In [19]:
if duplicate_count_before > 0:
    cleaned_df = cleaned_df.drop_duplicates().copy()

cleaned_df.shape

(3000000, 32)

## 5.16 Missing Value Audit

Missing values must not automatically be replaced with zero.

In this dataset, missing values can have business meaning.

For example:
- cancelled flights may not have actual departure or arrival times
- delay-cause fields may be missing when the cause is not applicable
- operational fields may be missing for cancelled or diverted flights

Therefore, missing values will be treated column by column.

In [20]:
missing_summary = (
    cleaned_df.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

CANCELLATION_CODE          2920860
DELAY_DUE_LATE_AIRCRAFT    2466137
DELAY_DUE_CARRIER          2466137
DELAY_DUE_SECURITY         2466137
DELAY_DUE_NAS              2466137
DELAY_DUE_WEATHER          2466137
ARR_DELAY                    86198
ELAPSED_TIME                 86198
AIR_TIME                     86198
WHEELS_ON                    79944
TAXI_IN                      79944
ARR_TIME                     79942
WHEELS_OFF                   78806
TAXI_OUT                     78806
DEP_DELAY                    77644
DEP_TIME                     77615
CRS_ELAPSED_TIME                14
DEST_CITY                        0
CRS_DEP_TIME                     0
DEST                             0
DOT_CODE                         0
FL_NUMBER                        0
ORIGIN                           0
ORIGIN_CITY                      0
AIRLINE                          0
AIRLINE_DOT                      0
FL_DATE                          0
AIRLINE_CODE                     0
DIVERTED            

In [21]:
missing_percentage = (
    cleaned_df.isna().mean() * 100
).sort_values(ascending=False)

missing_percentage

CANCELLATION_CODE          97.362000
DELAY_DUE_LATE_AIRCRAFT    82.204567
DELAY_DUE_CARRIER          82.204567
DELAY_DUE_SECURITY         82.204567
DELAY_DUE_NAS              82.204567
DELAY_DUE_WEATHER          82.204567
ARR_DELAY                   2.873267
ELAPSED_TIME                2.873267
AIR_TIME                    2.873267
WHEELS_ON                   2.664800
TAXI_IN                     2.664800
ARR_TIME                    2.664733
WHEELS_OFF                  2.626867
TAXI_OUT                    2.626867
DEP_DELAY                   2.588133
DEP_TIME                    2.587167
CRS_ELAPSED_TIME            0.000467
DEST_CITY                   0.000000
CRS_DEP_TIME                0.000000
DEST                        0.000000
DOT_CODE                    0.000000
FL_NUMBER                   0.000000
ORIGIN                      0.000000
ORIGIN_CITY                 0.000000
AIRLINE                     0.000000
AIRLINE_DOT                 0.000000
FL_DATE                     0.000000
A

## 5.17 Handling Cancellation Code

`CANCELLATION_CODE` is only meaningful for cancelled flights.

For flights that were not cancelled, a missing cancellation code means that no cancellation occurred.

We will therefore label those records as `NOT_CANCELLED`.

We will not invent a cancellation reason for cancelled flights where the code is actually missing.

In [22]:
cleaned_df.loc[
    (cleaned_df["CANCELLED"] == 0) &
    (cleaned_df["CANCELLATION_CODE"].isna()),
    "CANCELLATION_CODE"
] = "NOT_CANCELLED"

cleaned_df["CANCELLATION_CODE"].value_counts(dropna=False)

CANCELLATION_CODE
NOT_CANCELLED    2920860
B                  28772
D                  24417
A                  19476
C                   6475
Name: count, dtype: Int64

## 5.18 Cancellation Consistency Check

We will compare cancellation status with cancellation codes.

A non-cancelled flight should have the `NOT_CANCELLED` label after our business-rule transformation.

In [23]:
cancellation_check = (
    cleaned_df
    .groupby(["CANCELLED", "CANCELLATION_CODE"], dropna=False)
    .size()
    .reset_index(name="FLIGHT_COUNT")
)

cancellation_check

,CANCELLED,CANCELLATION_CODE,FLIGHT_COUNT
0,0.0,NOT_CANCELLED,2920860
1,1.0,A,19476
2,1.0,B,28772
3,1.0,C,6475
4,1.0,D,24417


## 5.19 Validating Binary Operational Flags

`CANCELLED` and `DIVERTED` should represent binary states.

Expected values:

- 0 → No
- 1 → Yes

In [24]:
cancelled_values = cleaned_df["CANCELLED"].dropna().unique()
diverted_values = cleaned_df["DIVERTED"].dropna().unique()

cancelled_values, diverted_values

(array([0., 1.]), array([0., 1.]))

## 5.20 Validating Physical Numeric Fields

Negative arrival or departure delays can be legitimate because a flight can arrive or depart early.

However, physical measurements such as:
- distance
- taxi time
- air time
- elapsed time

should not normally be negative.

We will identify such records instead of blindly deleting them.

In [25]:
physical_columns = [
    "TAXI_OUT",
    "TAXI_IN",
    "AIR_TIME",
    "DISTANCE",
    "ELAPSED_TIME",
    "CRS_ELAPSED_TIME"
]

negative_value_check = {
    column: (cleaned_df[column] < 0).sum()
    for column in physical_columns
}

negative_value_check

{'TAXI_OUT': np.int64(0),
 'TAXI_IN': np.int64(0),
 'AIR_TIME': np.int64(0),
 'DISTANCE': np.int64(0),
 'ELAPSED_TIME': np.int64(0),
 'CRS_ELAPSED_TIME': np.int64(0)}

## 5.21 Creating Total Delay Cause Minutes

The dataset contains several delay-cause columns:

- Carrier
- Weather
- NAS
- Security
- Late Aircraft

These fields may be missing when a cause is not applicable.

For analytical cause contribution, we will treat missing cause values as zero only while calculating the total cause contribution.

The original cause columns will remain unchanged.

In [26]:
delay_cause_columns = [
    "DELAY_DUE_CARRIER",
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT"
]

cleaned_df["TOTAL_DELAY_CAUSE_MINUTES"] = (
    cleaned_df[delay_cause_columns]
    .fillna(0)
    .sum(axis=1)
)

cleaned_df[
    ["ARR_DELAY", "TOTAL_DELAY_CAUSE_MINUTES"]
].head()

,ARR_DELAY,TOTAL_DELAY_CAUSE_MINUTES
0,-14.0,0.0
1,-5.0,0.0
2,0.0,0.0
3,24.0,24.0
4,-1.0,0.0


## 5.22 Creating a Business Classification Column

We will convert arrival delay into a business-friendly severity category.

Rules:

- <= 15 minutes → ON_TIME
- > 15 to 30 minutes → MODERATE_DELAY
- > 30 to 60 minutes → SIGNIFICANT_DELAY
- > 60 minutes → SEVERE_DELAY

Missing arrival delays will remain `UNKNOWN`.

In [27]:
conditions = [
    cleaned_df["ARR_DELAY"].isna(),
    cleaned_df["ARR_DELAY"] <= 15,
    cleaned_df["ARR_DELAY"] <= 30,
    cleaned_df["ARR_DELAY"] <= 60
]

choices = [
    "UNKNOWN",
    "ON_TIME",
    "MODERATE_DELAY",
    "SIGNIFICANT_DELAY"
]

cleaned_df["DELAY_SEVERITY"] = np.select(
    conditions,
    choices,
    default="SEVERE_DELAY"
)

cleaned_df["DELAY_SEVERITY"].value_counts(dropna=False)

DELAY_SEVERITY
ON_TIME              2398513
MODERATE_DELAY        184218
SEVERE_DELAY          175135
SIGNIFICANT_DELAY     155936
UNKNOWN                86198
Name: count, dtype: int64

## 5.23 Creating Date-Based Features

Datetime operations allow us to extract useful business dimensions from the flight date.

We will create:
- Year
- Month
- Day
- Day of week

These features will later support trend and seasonal analysis.

In [28]:
cleaned_df["YEAR"] = cleaned_df["FL_DATE"].dt.year
cleaned_df["MONTH"] = cleaned_df["FL_DATE"].dt.month
cleaned_df["DAY"] = cleaned_df["FL_DATE"].dt.day
cleaned_df["DAY_OF_WEEK"] = cleaned_df["FL_DATE"].dt.day_name()

cleaned_df[
    ["FL_DATE", "YEAR", "MONTH", "DAY", "DAY_OF_WEEK"]
].head()

,FL_DATE,YEAR,MONTH,DAY,DAY_OF_WEEK
0,2019-01-09,2019,1,9,Wednesday
1,2022-11-19,2022,11,19,Saturday
2,2022-07-22,2022,7,22,Friday
3,2023-03-06,2023,3,6,Monday
4,2020-02-23,2020,2,23,Sunday


## 5.24 Creating a Route Identifier

A route is defined as the combination of origin and destination airports.

This derived field will later be used for route-level delay and risk analysis.

In [29]:
cleaned_df["ROUTE"] = (
    cleaned_df["ORIGIN"] + " → " + cleaned_df["DEST"]
)

cleaned_df[
    ["ORIGIN", "DEST", "ROUTE"]
].head()

,ORIGIN,DEST,ROUTE
0,FLL,EWR,FLL → EWR
1,MSP,SEA,MSP → SEA
2,DEN,MSP,DEN → MSP
3,MSP,SFO,MSP → SFO
4,MCO,DFW,MCO → DFW


## 5.25 GroupBy Analysis

`groupby()` is one of the most important Pandas operations for business analytics.

It allows us to split data into groups and calculate metrics for each group.

We will calculate airline-level operational performance.

In [30]:
airline_group_summary = (
    cleaned_df
    .groupby("AIRLINE")
    .agg(
        TOTAL_FLIGHTS=("AIRLINE", "size"),
        AVG_DEP_DELAY=("DEP_DELAY", "mean"),
        AVG_ARR_DELAY=("ARR_DELAY", "mean")
    )
    .reset_index()
)

airline_group_summary

,AIRLINE,TOTAL_FLIGHTS,AVG_DEP_DELAY,AVG_ARR_DELAY
0,Alaska Airlines Inc.,100467,4.640018,1.097900
1,Allegiant Air,52738,13.907797,13.284601
2,American Airlines Inc.,383106,12.609895,6.661229
3,Delta Air Lines Inc.,395239,8.106748,1.085079
4,Endeavor Air Inc.,112463,5.951135,-1.258539
5,Envoy Air,121256,6.728256,3.278914
6,ExpressJet Airlines LLC d/b/a aha!,19082,12.774462,10.031976
7,Frontier Airlines Inc.,64466,16.033574,11.100429
8,Hawaiian Airlines Inc.,32114,5.089537,3.894757
9,Horizon Air,20634,4.832775,2.376242


## 5.26 Multiple Aggregations

We can calculate several business KPIs within the same grouped operation.

This provides a compact airline performance summary.

In [31]:
airline_performance = (
    cleaned_df
    .groupby("AIRLINE")
    .agg(
        TOTAL_FLIGHTS=("AIRLINE", "size"),
        AVG_DEP_DELAY=("DEP_DELAY", "mean"),
        AVG_ARR_DELAY=("ARR_DELAY", "mean"),
        MAX_ARR_DELAY=("ARR_DELAY", "max"),
        CANCELLED_FLIGHTS=("CANCELLED", "sum"),
        DIVERTED_FLIGHTS=("DIVERTED", "sum")
    )
    .reset_index()
)

airline_performance

,AIRLINE,TOTAL_FLIGHTS,AVG_DEP_DELAY,AVG_ARR_DELAY,MAX_ARR_DELAY,CANCELLED_FLIGHTS,DIVERTED_FLIGHTS
0,Alaska Airlines Inc.,100467,4.640018,1.097900,805.0,1934.0,239.0
1,Allegiant Air,52738,13.907797,13.284601,1736.0,2383.0,176.0
2,American Airlines Inc.,383106,12.609895,6.661229,2934.0,10907.0,981.0
3,Delta Air Lines Inc.,395239,8.106748,1.085079,1241.0,5982.0,782.0
4,Endeavor Air Inc.,112463,5.951135,-1.258539,2560.0,2394.0,221.0
5,Envoy Air,121256,6.728256,3.278914,1540.0,3633.0,311.0
6,ExpressJet Airlines LLC d/b/a aha!,19082,12.774462,10.031976,1844.0,1062.0,69.0
7,Frontier Airlines Inc.,64466,16.033574,11.100429,1416.0,1666.0,88.0
8,Hawaiian Airlines Inc.,32114,5.089537,3.894757,1642.0,388.0,28.0
9,Horizon Air,20634,4.832775,2.376242,757.0,374.0,23.0


In [32]:
airline_performance["CANCELLATION_RATE"] = (
    airline_performance["CANCELLED_FLIGHTS"]
    / airline_performance["TOTAL_FLIGHTS"]
) * 100

airline_performance["DIVERSION_RATE"] = (
    airline_performance["DIVERTED_FLIGHTS"]
    / airline_performance["TOTAL_FLIGHTS"]
) * 100

airline_performance

,AIRLINE,TOTAL_FLIGHTS,AVG_DEP_DELAY,AVG_ARR_DELAY,MAX_ARR_DELAY,CANCELLED_FLIGHTS,DIVERTED_FLIGHTS,CANCELLATION_RATE,DIVERSION_RATE
0,Alaska Airlines Inc.,100467,4.640018,1.097900,805.0,1934.0,239.0,1.925010,0.237889
1,Allegiant Air,52738,13.907797,13.284601,1736.0,2383.0,176.0,4.518563,0.333725
2,American Airlines Inc.,383106,12.609895,6.661229,2934.0,10907.0,981.0,2.846993,0.256065
3,Delta Air Lines Inc.,395239,8.106748,1.085079,1241.0,5982.0,782.0,1.513515,0.197855
4,Endeavor Air Inc.,112463,5.951135,-1.258539,2560.0,2394.0,221.0,2.128700,0.196509
5,Envoy Air,121256,6.728256,3.278914,1540.0,3633.0,311.0,2.996140,0.256482
6,ExpressJet Airlines LLC d/b/a aha!,19082,12.774462,10.031976,1844.0,1062.0,69.0,5.565454,0.361597
7,Frontier Airlines Inc.,64466,16.033574,11.100429,1416.0,1666.0,88.0,2.584308,0.136506
8,Hawaiian Airlines Inc.,32114,5.089537,3.894757,1642.0,388.0,28.0,1.208196,0.087189
9,Horizon Air,20634,4.832775,2.376242,757.0,374.0,23.0,1.812542,0.111467


## 5.27 Route-Level Analysis

The same grouping technique can be applied to routes.

We will identify route-level flight volume and average arrival delay.

In [33]:
route_summary = (
    cleaned_df
    .groupby("ROUTE")
    .agg(
        TOTAL_FLIGHTS=("ROUTE", "size"),
        AVG_ARR_DELAY=("ARR_DELAY", "mean"),
        MAX_ARR_DELAY=("ARR_DELAY", "max")
    )
    .reset_index()
)

route_summary.head()

,ROUTE,TOTAL_FLIGHTS,AVG_ARR_DELAY,MAX_ARR_DELAY
0,ABE → ATL,398,6.190840,933.0
1,ABE → BNA,48,16.181818,258.0
2,ABE → CLT,483,-14.505330,325.0
3,ABE → DEN,1,-7.000000,-7.0
4,ABE → DTW,292,4.780488,977.0


## 5.28 Airport-Level Analysis

We can also aggregate flights by origin airport.

This will later support airport operational risk analysis.

In [34]:
airport_summary = (
    cleaned_df
    .groupby("ORIGIN")
    .agg(
        TOTAL_DEPARTURES=("ORIGIN", "size"),
        AVG_DEP_DELAY=("DEP_DELAY", "mean"),
        AVG_ARR_DELAY=("ARR_DELAY", "mean")
    )
    .reset_index()
)

airport_summary.head()

,ORIGIN,TOTAL_DEPARTURES,AVG_DEP_DELAY,AVG_ARR_DELAY
0,ABE,2023,9.487076,1.535333
1,ABI,901,8.267735,6.550343
2,ABQ,9284,8.797039,3.598339
3,ABR,332,14.435976,7.625000
4,ABY,436,2.856148,-2.437209


## 5.29 Pivot Tables

A pivot table summarizes data across multiple dimensions.

We will create a monthly airline arrival-delay summary.

This structure will later be useful for visualization and Tableau.

In [35]:
monthly_airline_delay = pd.pivot_table(
    cleaned_df,
    values="ARR_DELAY",
    index="MONTH",
    columns="AIRLINE",
    aggfunc="mean"
)

monthly_airline_delay.head()

AIRLINE,Alaska Airlines Inc.,Allegiant Air,American Airlines Inc.,Delta Air Lines Inc.,Endeavor Air Inc.,Envoy Air,ExpressJet Airlines LLC d/b/a aha!,Frontier Airlines Inc.,Hawaiian Airlines Inc.,Horizon Air,JetBlue Airways,Mesa Airlines Inc.,PSA Airlines Inc.,Republic Airline,SkyWest Airlines Inc.,Southwest Airlines Co.,Spirit Air Lines,United Air Lines Inc.
MONTH,,,,,,,,,,,,,,,,,,
1,1.499943,9.611672,2.165527,-0.988837,-0.471555,3.592616,6.993609,5.421808,2.342327,1.164442,8.965926,4.880086,4.723915,-0.471583,5.373782,-0.818042,6.059159,3.203556
2,3.082346,10.355121,5.574381,0.462834,-0.472713,5.392232,8.873185,7.455273,2.979267,2.942662,10.356878,7.198558,6.709389,1.083348,7.345776,-0.126483,8.039647,2.642383
3,-0.587647,11.335202,4.296357,-0.057361,-3.001876,0.707142,1.428319,9.691390,4.964858,0.110475,7.990091,2.554221,1.961454,-1.825979,1.459668,1.857041,8.232472,2.551378
4,0.123157,14.350942,5.312337,1.073820,-1.906257,1.030616,8.198987,12.549727,4.681021,0.518980,13.885837,1.844339,0.760450,-0.451719,0.276750,3.570605,13.411494,5.026521
5,-1.180531,9.051544,6.912361,-0.291619,-3.243735,3.434450,10.354724,12.869247,5.043223,1.216658,6.842789,7.430026,1.238052,-0.668582,1.493735,3.210588,8.110955,6.153626


## 5.30 Merging Analytical Tables

`merge()` combines two DataFrames using a common key.

We will merge airline performance metrics with airline flight-volume information.

This demonstrates how separate analytical tables can be combined for reporting.

In [36]:
airline_volume = (
    cleaned_df
    .groupby("AIRLINE")
    .size()
    .reset_index(name="FLIGHT_VOLUME")
)

airline_delay_metrics = (
    cleaned_df
    .groupby("AIRLINE")
    .agg(
        AVG_ARR_DELAY=("ARR_DELAY", "mean"),
        AVG_DEP_DELAY=("DEP_DELAY", "mean")
    )
    .reset_index()
)

In [37]:
airline_merged = pd.merge(
    airline_volume,
    airline_delay_metrics,
    on="AIRLINE",
    how="inner"
)

airline_merged

,AIRLINE,FLIGHT_VOLUME,AVG_ARR_DELAY,AVG_DEP_DELAY
0,Alaska Airlines Inc.,100467,1.097900,4.640018
1,Allegiant Air,52738,13.284601,13.907797
2,American Airlines Inc.,383106,6.661229,12.609895
3,Delta Air Lines Inc.,395239,1.085079,8.106748
4,Endeavor Air Inc.,112463,-1.258539,5.951135
5,Envoy Air,121256,3.278914,6.728256
6,ExpressJet Airlines LLC d/b/a aha!,19082,10.031976,12.774462
7,Frontier Airlines Inc.,64466,11.100429,16.033574
8,Hawaiian Airlines Inc.,32114,3.894757,5.089537
9,Horizon Air,20634,2.376242,4.832775


## 5.31 Merge Validation

After merging two analytical tables, we should validate:

- Number of airlines
- Missing keys
- Unexpected row multiplication

The merged table should contain one row per airline.

airline_merged

In [38]:
airline_merged.shape

(18, 4)

In [39]:
airline_merged["AIRLINE"].nunique()

18

# 5.32 Final Data Cleaning Validation

Before saving the cleaned dataset, we will perform a final validation.

We will check:

- Shape
- Duplicate records
- Missing values
- Data types
- Important derived columns
- Binary flags

In [40]:
cleaning_validation = {
    "ROWS": len(cleaned_df),
    "COLUMNS": cleaned_df.shape[1],
    "DUPLICATES": cleaned_df.duplicated().sum(),
    "MISSING_FL_DATE": cleaned_df["FL_DATE"].isna().sum(),
    "INVALID_CANCELLED_VALUES": (
        ~cleaned_df["CANCELLED"].isin([0, 1])
    ).sum(),
    "INVALID_DIVERTED_VALUES": (
        ~cleaned_df["DIVERTED"].isin([0, 1])
    ).sum()
}

cleaning_validation

{'ROWS': 3000000,
 'COLUMNS': 39,
 'DUPLICATES': np.int64(0),
 'MISSING_FL_DATE': np.int64(0),
 'INVALID_CANCELLED_VALUES': np.int64(0),
 'INVALID_DIVERTED_VALUES': np.int64(0)}

In [41]:
final_missing_report = (
    cleaned_df.isna()
    .sum()
    .to_frame("MISSING_COUNT")
)

final_missing_report["MISSING_PERCENT"] = (
    final_missing_report["MISSING_COUNT"]
    / len(cleaned_df)
) * 100

final_missing_report

,MISSING_COUNT,MISSING_PERCENT
FL_DATE,0,0.000000
AIRLINE,0,0.000000
AIRLINE_DOT,0,0.000000
AIRLINE_CODE,0,0.000000
DOT_CODE,0,0.000000
FL_NUMBER,0,0.000000
ORIGIN,0,0.000000
ORIGIN_CITY,0,0.000000
DEST,0,0.000000
DEST_CITY,0,0.000000


In [42]:
cleaned_df.head()

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,TOTAL_DELAY_CAUSE_MINUTES,DELAY_SEVERITY,YEAR,MONTH,DAY,DAY_OF_WEEK,ROUTE
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,NaN,NaN,NaN,0.0,ON_TIME,2019,1,9,Wednesday,FLL → EWR
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,NaN,NaN,NaN,0.0,ON_TIME,2022,11,19,Saturday,MSP → SEA
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,NaN,NaN,NaN,0.0,ON_TIME,2022,7,22,Friday,DEN → MSP
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,24.0,0.0,0.0,24.0,MODERATE_DELAY,2023,3,6,Monday,MSP → SFO
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,NaN,NaN,NaN,0.0,ON_TIME,2020,2,23,Sunday,MCO → DFW


In [43]:
cleaned_df.shape

(3000000, 39)

# 5.33 Saving the Cleaned Dataset

The raw dataset remains unchanged in `df`.

The validated dataset is stored in `cleaned_df`.

We will save the cleaned dataset separately so that future analysis can use the cleaned version without modifying the raw source.

In [44]:
import os

clean_path = "data/cleaned_data/cleaned_flight_delay.csv"

os.makedirs(os.path.dirname(clean_path), exist_ok=True)

cleaned_df.to_csv(
    clean_path,
    index=False
)

clean_path

'data/cleaned_data/cleaned_flight_delay.csv'

## 5.34 Verifying the Saved Dataset

We will read a small sample from the saved file to confirm that the output file was created successfully.

In [45]:
cleaned_check = pd.read_csv(
    clean_path,
    nrows=5
)

cleaned_check

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,TOTAL_DELAY_CAUSE_MINUTES,DELAY_SEVERITY,YEAR,MONTH,DAY,DAY_OF_WEEK,ROUTE
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,NaN,NaN,NaN,0.0,ON_TIME,2019,1,9,Wednesday,FLL → EWR
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,NaN,NaN,NaN,0.0,ON_TIME,2022,11,19,Saturday,MSP → SEA
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,NaN,NaN,NaN,0.0,ON_TIME,2022,7,22,Friday,DEN → MSP
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,24.0,0.0,0.0,24.0,MODERATE_DELAY,2023,3,6,Monday,MSP → SFO
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,NaN,NaN,NaN,0.0,ON_TIME,2020,2,23,Sunday,MCO → DFW


# Phase 5 Validation

Pandas topics completed:

- Series
- DataFrame
- Column selection
- `loc`
- `iloc`
- Filtering
- `isin()`
- `between()`
- Sorting
- Data copying
- Column standardization
- Data-type conversion
- Text cleaning
- Duplicate validation
- Missing-value analysis
- Business-rule missing-value treatment
- Conditional column creation
- Date feature extraction
- String-based route creation
- GroupBy
- Aggregation
- Multiple aggregations
- Airline-level analysis
- Route-level analysis
- Airport-level analysis
- Pivot table
- Merge
- Merge validation
- Data validation
- Saving cleaned data

In [46]:
phase_5_topics = {
    "Series": True,
    "DataFrame": True,
    "Selection": True,
    "Filtering": True,
    "Sorting": True,
    "Cleaning": True,
    "Missing Values": True,
    "Data Types": True,
    "GroupBy": True,
    "Aggregation": True,
    "Merge": True,
    "Pivot Table": True,
    "Validation": True,
    "Clean Dataset Saved": True
}

phase_5_topics

{'Series': True,
 'DataFrame': True,
 'Selection': True,
 'Filtering': True,
 'Sorting': True,
 'Cleaning': True,
 'Missing Values': True,
 'Data Types': True,
 'GroupBy': True,
 'Aggregation': True,
 'Merge': True,
 'Pivot Table': True,
 'Validation': True,
 'Clean Dataset Saved': True}